# 第 07 章　高级函数

> **本章定位**：把"函数"当作数据来用。同一件事可以写 `for` 循环，也可以一行 `map` / `filter` / `sorted` 完成；统一给一批函数加日志，则用一个装饰器解决。
>
> **知识点范围**：匿名函数 `lambda`、`map()`、`filter()`、`sorted(key=)` 与多级排序、`zip()`、高阶函数（函数作参数 / 返回值）、装饰器（`*args` `**kwargs` 透传、`@` 语法糖、`__name__`）。
>
> **本章约定**：可用前 6 章全部内容；**不使用 `import`**（标准库在第 10 章）。因此装饰器示例统计"调用次数"而非"耗时"，避免依赖 `time` 模块。

---

## 本章学习路径

| 阶段 | 内容 | 编号 | 作用 |
|---|---|---|---|
| 第一部分 | 知识讲解 | 知识点 7.1 — 7.5 | 函数式写法与装饰器 + 随堂示例 |
| 第二部分 | 代码示范 | 案例 7-1、案例 7-2 | 批处理 / 统一日志 |
| 第三部分 | 布置练习 | 练习 7-1、练习 7-2 | 超速配对 / 多关键字排序 |
| 第四部分 | 习题答案 | 答案 7-1、答案 7-2 | 对照解析与评分要点 |

**知识点与代码对照表**：

| 知识点 | 名称 | 讲解位置 | 代码示范 | 对应练习 |
|---|---|---|---|---|
| 7.1 | `lambda` 匿名函数 | 7.2 | 案例 7-1 | 练习 7-1、练习 7-2 |
| 7.2 | `map()` 与 `filter()` | 7.3 | 案例 7-1 | 练习 7-1 |
| 7.3 | `sorted(key=)` 与多级排序 | 7.4 | 案例 7-1 | 练习 7-2 |
| 7.4 | `zip()` 配对 | 7.5 | —— | 练习 7-1 |
| 7.5 | 装饰器与高阶函数 | 7.6 | 案例 7-2 | —— |

## 教学提示（教师用）

- **建议课时**：4 学时（2 次课），对应课程标准模块 7。
- **讲解主线**：把"循环 + 判断"升级为"声明式表达"：`map` 管转换、`filter` 管筛选、`sorted(key=)` 管排序，三者各配一个 `lambda`；装饰器只讲一句核心——"不修改原函数，也能加行为"。
- **学生常见卡点**：`map` / `filter` 返回迭代器，不套 `list()` 看不到内容，且只能遍历一次；`sorted` 与 `sort` 混淆；`lambda` 体内不能写语句；多级排序的元组 key 与负号取反容易写错。
- **课堂活动建议**：把 7-1 中的 `lambda` 逐个改写为 `def`，讨论"什么时候该用具名函数"，避免学生陷入"越短越好"的误解。

---

# 第一部分　知识讲解

## 7.1 场景导入：十几万条记录怎么处理

车载终端每秒上报一次数据，一天产生十几万条记录。用 `for` 逐条处理当然可行，但"筛出超速车辆""把 km/h 换算成 m/s""按车速排名"这三类操作有更紧凑的表达——把**函数当作参数**传给 `map` / `filter` / `sorted`，一行完成。

而"给所有采集函数统一加上日志"，逐个改函数体既不现实也易错，正确的工具是**装饰器**：在不改动原函数的前提下，为它套上一层"前后处理"。

## 7.2 知识点 7.1　`lambda` 匿名函数

### 7.2.1 核心概念

`lambda` 用于定义**只用一次、没有名字**的小函数，形式为：

```text
lambda 参数列表: 表达式
```

它等价于一个只有 `return` 的 `def`：

| 写法 | 等价 `def` |
|---|---|
| `f = lambda x: x * 2` | `def f(x): return x * 2` |
| `lambda a, b: a + b` | `def f(a, b): return a + b` |

**原理补充（概念深化）**：`lambda` 是表达式而不是语句：解释器执行到它就现场创建一个函数对象并把它作为结果返回，所以能直接写在参数位置（如 `key=lambda...`）；它和 `def` 编译出的都是同一种函数对象，区别只在 `lambda` 体被语法限定为「单个表达式」，表达式的值自动成为返回值，因此既不需要也不能写 `return`。为什么不能写语句？赋值、`print` 语句、`if` 块都不产生值，而 lambda 体内只允许求值表达式；想表达分支只能用三元式 `a if cond else b`。`lambda` 不是「更快的函数」也不更高级，它只是「没有名字、用完即弃」的简写：其 `__name__` 恒为 `<lambda>`，出错回溯里看不出用途。常见误解是把它赋给变量 `f = lambda x: ...` 就算「起了名」，其实反而丢掉了可读性这一唯一优势；一旦逻辑需要复用、命名或调试，就该写具名 `def`。

### 7.2.2 语法要点

- **只能写一个表达式**，不能写语句（不能赋值、不能 `print`、不能 `if-else` 语句块，但可用三元表达式 `a if cond else b`）。
- 冒号后就是返回值，**不需要写 `return`**。
- 常作为 `map` / `filter` / `sorted` 的 `key` 参数"就地"使用，用完即弃。
- 可赋值给变量，但**不推荐**：需要命名说明用途时，应写 `def`。

### 7.2.3 常见用法与易错点

| 易错点 | 现象 | 修正 |
|---|---|---|
| 写成多行语句 | `SyntaxError` | 复杂逻辑改用 `def` |
| 加 `return` | `SyntaxError: invalid syntax` | `lambda x: x + 1`，无 `return` |
| 无参数时写成 `lambda:` | 合法（返回常量），但通常无意义 | 确认是否真需要 |
| 在 `lambda` 里 `print` | 不报错但返回 `None` | 调试用 `def`，别用 `lambda` |

### 7.2.4 随堂示例：lambda 就是没有名字的小函数

**核心要点**：验证 `lambda 参数: 表达式` 只能写一个表达式、自动返回结果，常作为 `key=` 或 `map()` 的参数就地使用。

In [ ]:
to_ms = lambda v: round(v / 3.6, 1)
print("72 km/h →", to_ms(72), "m/s")
print("108 km/h →", to_ms(108), "m/s")

pairs = [("京A", 62), ("京B", 132), ("津C", 78)]
print("按车速降序：", sorted(pairs, key=lambda p: p[1], reverse=True))

# 等价的 def 写法，对比可读性
def to_ms2(v):
    return round(v / 3.6, 1)

print("两种写法结果一致：", to_ms(72) == to_ms2(72))

**预期输出**：

```text
72 km/h → 20.0 m/s
108 km/h → 30.0 m/s
按车速降序： [('京B', 132), ('津C', 78), ('京A', 62)]
两种写法结果一致： True
```

**变式代码（迁移练习）**：相对随堂示例用 lambda 做数值换算，本变式在 lambda 体内用三元表达式表达分支，并直接作为 `sorted` 的 `key` 就地使用，验证「单表达式」边界与匿名函数 `__name__` 恒为 `<lambda>`。

In [ ]:
# lambda 体内用三元表达式实现分支，仍是单个表达式
level = lambda v: "超速" if v > 120 else "正常"
print("单次调用：", level(132), level(78))
speeds = [62, 132, 78, 150]
# lambda 作为 sorted 的 key 就地使用：超速(0)排前，组内再按车速
ranked = sorted(speeds, key=lambda v: (0 if v > 120 else 1, v))
print("超速优先的排名：", ranked)
print("匿名函数名：", level.__name__)

**预期输出**：

```text
单次调用： 超速 正常
超速优先的排名： [132, 150, 62, 78]
匿名函数名： <lambda>
```

### 7.2.5 实操步骤（上机跟随）

1. `level(132)` 返回 `超速`、`level(78)` 返回 `正常`，说明三元表达式 `a if 条件 else b` 是表达式、可放进 lambda
2. `key=lambda v: (0 if v > 120 else 1, v)` 把超速车映射到 `(0, v)`、其余映射到 `(1, v)`，输出 `[132, 150, 62, 78]`
3. 打印 `level.__name__` 得到 `<lambda>`，验证 lambda 没有有意义的名字
4. 试着在 lambda 里写 `x = v + 1` 语句，预期 `SyntaxError`，说明体内只能是表达式
5. 把三元分支改写为等价 `def`，结果与 lambda 一致，说明二者是同一种函数对象

### 7.2.6 常见问题 FAQ

| 现象 | 根因 | 处理 |
|---|---|---|
| lambda 里写赋值/语句报 `SyntaxError` | lambda 体只允许单个表达式，语句不产生值 | 分支改用三元表达式 `a if cond else b`，复杂逻辑改 `def` |
| 在 lambda 里加 `return` 报 `invalid syntax` | 表达式的值自动作为返回值，不能写 `return` | 直接写表达式，冒号后不出现 `return` |
| 回溯里函数名叫 `<lambda>`，看不出用途 | lambda 的 `__name__` 恒为 `<lambda>` | 需要命名、复用、调试时改写具名 `def` |
| `f = lambda v: print(v)` 调用后拿到 `None` | `print` 表达式返回 `None`，被当作 lambda 的返回值 | 要可见输出应在外部 `print(f(v))`，调试别用 lambda |

## 7.3 知识点 7.2　`map()` 与 `filter()`

### 7.3.1 核心概念

| 函数 | 作用 | 参数顺序 | 结果 |
|---|---|---|---|
| `map(f, 序列)` | 对每个元素应用 `f`，**变换** | 函数在前，序列在后 | 迭代器 |
| `filter(f, 序列)` | 保留使 `f` 为真的元素，**筛选** | 同上 | 迭代器 |

**关键**：两者返回的都是**迭代器**，不是列表。迭代器只能遍历一次，直接 `print` 会显示 `<map object at 0x…>`，必须用 `list()` 转换才能看到内容。

**原理补充（概念深化）**：`map` 与 `filter` 返回的都不是列表，而是惰性迭代器：对象内部只保存「函数 + 输入迭代器」的引用，每次 `next()` 才对下一个元素应用 `f`、产出一个结果。这既省内存（不必先生成完整中间列表），也解释了「只能遍历一次」——迭代器有单向游标，走到末尾就耗尽，第二次 `list()` 得到空表。参数顺序为何是函数在前？`map(f, seq)` 把「做什么」放前、「对谁做」放后，与 `filter(f, seq)` 保持一致；二者语义正交：`map` 逐元素变换、个数不变，`filter` 用 `f` 的真值决定去留、个数可能减少。常见误解是以为 `map(...)` 已经「算完了」，其实直接 `print` 只显示 `<map object ...>` 这样的对象描述；要复用结果必须先 `list(...)` 固化成列表。迭代器惰性也意味着 `f` 里对外部状态的副作用会延迟到真正遍历时才发生。

### 7.3.2 语法要点

| 写法 | 结果（speeds = [62, 132, 150]） |
|---|---|
| `map(lambda v: v / 3.6, speeds)` | 迭代器，非列表 |
| `list(map(lambda v: v / 3.6, speeds))` | `[17.2, 36.7, 41.7]` |
| `list(filter(lambda v: v > 120, speeds))` | `[132, 150]` |
| `list(map(str, speeds))` | `['62', '132', '150']`（`f` 可为内置函数） |

### 7.3.3 常见用法与易错点

| 易错点 | 现象 | 修正 |
|---|---|---|
| 忘记 `list()` | 打印出 `<map object …>` | `list(map(...))` |
| 二次使用同一个迭代器 | 第二次为空 | 先转列表再复用 |
| `filter` 用成 `map` | 结果个数不变 | 筛选要用 `filter` |
| 参数顺序写反 | `TypeError` | 函数在前、序列在后 |

> **对比第 5 章**：`list(filter(lambda v: v > 120, speeds))` 与 `[v for v in speeds if v > 120]` 结果相同。选择依据：逻辑简单、需要链式组合时用推导式更清晰；已有现成函数时用 `map` 更省事。

### 7.3.4 随堂示例：map 转换、filter 筛选

**核心要点**：验证 `map()` 对每个元素做同一变换、`filter()` 只保留条件为真的元素；两者都返回迭代器，必须套 `list()` 才能看到内容。

In [ ]:
speeds = [62, 118, 132, 78]

ms = list(map(lambda v: round(v / 3.6, 1), speeds))
print("换算 m/s：", ms)

over = list(filter(lambda v: v > 120, speeds))
print("超速车速：", over, "| 辆数：", len(over))
print("超速率：", round(len(over) / len(speeds) * 100, 1), "%")

# 不套 list() 时拿到的是迭代器：只占位置、还没算
lazy = map(str, speeds)
print("不套 list() 的类型：", type(lazy))
print("套上 list() 才有内容：", list(lazy))

**预期输出**：

```text
换算 m/s： [17.2, 32.8, 36.7, 21.7]
超速车速： [132] | 辆数： 1
超速率： 25.0 %
不套 list() 的类型： <class 'map'>
套上 list() 才有内容： ['62', '118', '132', '78']
```

**变式代码（迁移练习）**：相对随堂示例分别对全量车速做 map 与 filter，本变式迁移为「filter 的结果再喂给 map」的链式处理，并用同一 map 对象遍历两次验证迭代器只能消费一次。

In [ ]:
# 用一批检测器上报车速演示 filter 筛选与 map 变换
speeds = [62, 132, 78, 150, 95, 118]
over = list(filter(lambda v: v > 120, speeds))        # 筛选：个数减少
labels = list(map(lambda v: str(v) + "限速提醒", over))  # 变换：个数不变
print("超速车速：", over, "共", len(over), "条")
print("提醒文本：", labels)
lazy = map(lambda v: round(v / 3.6, 1), speeds)
first = list(lazy)                         # 迭代器第一次遍历即耗尽
print("首次遍历：", first, "| 原对象类型：", type(lazy).__name__)
print("二次遍历：", list(lazy))

**预期输出**：

```text
超速车速： [132, 150] 共 2 条
提醒文本： ['132限速提醒', '150限速提醒']
首次遍历： [17.2, 36.7, 21.7, 41.7, 26.4, 32.8] | 原对象类型： map
二次遍历： []
```

### 7.3.5 实操步骤（上机跟随）

1. `filter(lambda v: v > 120, ...)` 只保留 132、150，输出 `共 2 条`，验证筛选会减少元素个数
2. 把 `over` 再传给 `map` 得到 `['132限速提醒', '150限速提醒']`，个数仍为 2，验证变换不改变数量
3. `lazy = map(...)` 不套 `list()` 时 `type` 为 `map`，此时没有任何元素被真正计算
4. `first = list(lazy)` 得到 6 个换算值，是第一次也是唯一一次有效遍历
5. 再次 `list(lazy)` 输出 `[]`，验证迭代器游标已到末尾、不能重复消费

### 7.3.6 常见问题 FAQ

| 现象 | 根因 | 处理 |
|---|---|---|
| 打印出 `<map object at 0x…>` | map/filter 返回惰性迭代器，尚未计算 | 套 `list(map(...))` 固化后再打印 |
| 同一个迭代器第二次用变成空列表 | 迭代器单向游标走到底即耗尽 | 把第一次结果 `list()` 存成列表再复用 |
| 筛选后结果个数没变 | 把 `filter` 误写成 `map`，map 会保留全部元素 | 筛选条件用 `filter`，变换用 `map` |
| `TypeError` 或结果全错 | 把参数写成 `map(序列, 函数)`，顺序反了 | 固定为 `map(函数, 序列)`、`filter(函数, 序列)` |

## 7.4 知识点 7.3　`sorted(key=)` 与多级排序

### 7.4.1 核心概念

`sorted(序列, key=函数, reverse=False)` 返回新列表，`key` 指定"按什么排序"：

- `key` 接收一个函数，该函数把每个元素"映射"为比较依据；
- `reverse=True` 表示降序；
- 原列表**不变**。

**多级排序**：`key` 返回**元组**时，Python 按元组元素依次比较——先用第一个元素，相同再用第二个。

**原理补充（概念深化）**：`sorted` 采用 DSU（装饰—排序—去装饰）：先对序列每个元素调用一次 `key` 函数得到「比较键」并与原元素配对，再只按键排序，最后剥掉键还原元素。因此 `key` 必须传「函数本身」（`key=len`），写成 `key=len()` 等于先调用一次、把返回值当参数，自然报 `TypeError`；每个元素的键只算一次，这也比两两比较的 `cmp` 更高效。元组 key 能做多级排序，是因为 Python 对元组按字典序比较：先比第 0 项，相等再比第 1 项。想让「某列降序、另一列升序」混排时不能整体 `reverse`（会把两列一起翻），于是对数值列取负号 `-r[2]`，把降序折算进升序比较。`sorted` 还是稳定排序，键相等的元素保持原相对次序，这是「两次 sorted 分主次键」可行的原因；它永远返回新列表，原列表不变，别与原地方法 `lst.sort()`（返回 `None`）混淆。

### 7.4.2 语法要点

| 需求 | 写法 | 说明 |
|---|---|---|
| 按车速降序 | `sorted(speeds, key=lambda v: v, reverse=True)` | 元素本身即键时可省略 `key` |
| 按排队长度降序 | `sorted(roads, key=lambda r: r[2], reverse=True)` | 取元组第 3 个字段 |
| 车速升序、同速排队降序 | `sorted(roads, key=lambda r: (r[1], -r[2]))` | 元组 key + 负号实现"第二列降序" |
| 按字符串长度排 | `sorted(names, key=len)` | `key` 可为内置函数 |

### 7.4.3 常见用法与易错点

| 易错点 | 现象 | 修正 |
|---|---|---|
| 写成 `key=len()` | `TypeError` | `key` 传**函数本身**，不带括号 |
| 想降序却只对第一列生效 | 第二列仍升序 | 用负号，或用两次 `sorted`（稳定排序） |
| 对元组用 `-r[2]` 但字段是字符串 | `TypeError` | 负号只能用于数值 |
| 以为 `sorted` 改了原列表 | 原列表不变 | 需要原地改就用 `lst.sort(key=...)` |

### 7.4.4 随堂示例：多关键字排序：元组 key 与负号

**核心要点**：验证 `sorted(key=...)` 的 key 返回元组时按元组顺序逐级比较，以及用 `-` 取反实现「某一列降序、另一列升序」的混排。

In [ ]:
roads = [("京港澳", 45, 320), ("京开", 62, 80), ("机场", 45, 40)]

by_queue = sorted(roads, key=lambda r: r[2], reverse=True)
print("按排队降序：", by_queue)

# 车速升序；车速相同时排队长的在前
by_speed = sorted(roads, key=lambda r: (r[1], -r[2]))
print("车速升序+排队降序：", by_speed)

print("最拥堵：", by_queue[0][0], by_queue[0][2], "m")
print("原列表未被修改：", roads)

**预期输出**：

```text
按排队降序： [('京港澳', 45, 320), ('京开', 62, 80), ('机场', 45, 40)]
车速升序+排队降序： [('京港澳', 45, 320), ('机场', 45, 40), ('京开', 62, 80)]
最拥堵： 京港澳 320 m
原列表未被修改： [('京港澳', 45, 320), ('京开', 62, 80), ('机场', 45, 40)]
```

**变式代码（迁移练习）**：相对随堂示例用二元组 key 混排车速与排队，本变式迁移为「等级分 + 排队」的三字段元组 key，并追加 `key=len`（函数本身）与「原列表不变」的校验，验证 DSU 机制与 key 不能带括号。

In [ ]:
# 三字段路段：(名称, 拥堵等级分, 排队长度)，等级分越小越优先
roads = [("京港澳", 3, 320), ("京开", 1, 80), ("机场", 3, 40), ("京承", 1, 120)]
# 元组 key：先按等级升序，同级按排队降序（负号折算）
ranked = sorted(roads, key=lambda r: (r[1], -r[2]))
print("等级优先+排队降序：", ranked)
names = [r[0] for r in ranked]            # 只取排序后的路段名
print("处置顺序：", names)
print("按名称长度排：", sorted(names, key=len))
print("原列表未变：", roads[0])

**预期输出**：

```text
等级优先+排队降序： [('京承', 1, 120), ('京开', 1, 80), ('京港澳', 3, 320), ('机场', 3, 40)]
处置顺序： ['京承', '京开', '京港澳', '机场']
按名称长度排： ['京承', '京开', '机场', '京港澳']
原列表未变： ('京港澳', 3, 320)
```

### 7.4.5 实操步骤（上机跟随）

1. `key=lambda r: (r[1], -r[2])` 先比等级分（1 在前），同级用 `-r[2]` 把排队降序折算进升序，得到京承、京开、京港澳、机场
2. 等级分同为 1 时，京承排队 120 大于京开 80，因 `-120 < -80` 而排前，验证负号取反
3. `sorted(names, key=len)` 得到 `['京承', '京开', '机场', '京港澳']`，注意 `key` 传函数本身 `len` 而非 `len()`
4. 把 `key` 改成 `key=len()` 运行，预期 `TypeError`，验证 key 必须是可调用对象
5. 最后打印 `roads[0]` 仍是 `('京港澳', 3, 320)`，验证 `sorted` 返回新列表、不改原列表

### 7.4.6 常见问题 FAQ

| 现象 | 根因 | 处理 |
|---|---|---|
| `key=len()` 报 `TypeError` | key 被写成「调用一次的结果」，而不是函数对象 | 去掉括号写 `key=len`，让 sorted 对每个元素调用 |
| 第二列仍是升序、混排失败 | 元组 key 里没对降序列取负号 | 数值列写 `-r[2]`，或分两次稳定 `sorted` |
| `TypeError: bad operand type for unary -` | 对字符串字段用了负号 | 负号只用于数值；字符串列降序改两次稳定排序 |
| 排序后原列表也被改了 | 误用 `lst.sort()`（原地、返回 `None`） | 需要新列表就用 `sorted()` |

## 7.5 知识点 7.4　`zip()` 配对

### 7.5.1 核心概念

`zip(a, b)` 把两个序列**按下标一一配对**成元组，返回迭代器。常用于"车牌列表 + 车速列表"合并成记录列表。

| 写法 | 结果 |
|---|---|
| `list(zip(["京A", "京B"], [62, 118]))` | `[('京A', 62), ('京B', 118)]` |
| `dict(zip(["京A", "京B"], [62, 118]))` | `{'京A': 62, '京B': 118}` |

**原理补充（概念深化）**：`zip(a, b)` 返回一个迭代器，内部持有对各输入序列的迭代器；每次 `next()` 从每个序列各取一个元素组装成元组，只要任一序列耗尽就立即抛 `StopIteration` 结束。可见「以最短为准」并非容错设计，而是迭代协议的必然结果：长序列多出的元素根本没机会被取用，Python 也不会报错提示丢弃，所以这是最隐蔽的数据丢失来源，配对前务必 `len()` 校验。`dict(zip(k, v))` 之所以成立，是因为 `dict()` 接受「可迭代的键值对」，而 zip 恰好产出二元组序列，键列表与值列表便自然拼成字典。内存视角上，只有 `list(zip(...))` 才一次性物化全部元组，在此之前 zip 对象只占固定小空间。常见误解是把 zip 当成「两两组合/笛卡尔积」，其实它只做「对齐配对」，要组合得用嵌套循环。

### 7.5.2 语法要点

- 配对数以**最短序列**为准，长度不一致时**静默截断**，不报错——这是最隐蔽的 bug 来源，配对前应 `len()` 校验。
- `zip` 返回迭代器，需要列表就 `list(zip(...))`。
- 可用于三个及以上序列：`zip(plates, speeds, lanes)`。

### 7.5.3 常见用法与易错点

| 易错点 | 现象 | 修正 |
|---|---|---|
| 长度不一致 | 少了几条记录，无报错 | 先 `len(a) == len(b)` 校验 |
| 直接打印 | 显示 `<zip object …>` | `list(zip(...))` |
| 复用已耗尽的 `zip` | 第二次为空 | 转成列表保存 |
| 误以为会做笛卡尔积 | 得到的是配对而非组合 | 组合需嵌套循环 |

### 7.5.4 随堂示例：zip 把两个列表配对

**核心要点**：验证 `zip()` 按位置把多个序列打包成元组，长度以最短的为准；配合 `for` 可同时取出车牌与车速。

In [ ]:
plates = ["京A12345", "京B67890", "津C24680"]
speeds = [62, 132, 78]

pairs = list(zip(plates, speeds))
print("配对结果：", pairs)
print("配对数：", len(pairs))

for plate, speed in pairs:
    if speed > 120:                  # 两个分支各自输出，不带多余空格
        print(plate, "→", speed, "km/h（超速）")
    else:
        print(plate, "→", speed, "km/h")

**预期输出**：

```text
配对结果： [('京A12345', 62), ('京B67890', 132), ('津C24680', 78)]
配对数： 3
京A12345 → 62 km/h
京B67890 → 132 km/h（超速）
津C24680 → 78 km/h
```

**变式代码（迁移练习）**：相对随堂示例用两个等长序列配对，本变式迁移为三序列 zip、`dict(zip(...))` 成字典，并故意让第二序列变短，验证「以最短为准静默截断」及配对前长度校验的必要性。

In [ ]:
plates = ["京A12345", "京B67890", "津C24680", "冀D11223"]
speeds = [62, 132, 78, 150]
lanes = [2, 3, 1, 2]
# 三序列 zip：按下标组装成 (车牌, 车速, 车道)
records = list(zip(plates, speeds, lanes))
print("配对记录：", records)
print("车牌→车速字典：", dict(zip(plates, speeds)))
print("长度校验：", len(plates) == len(speeds) == len(lanes))
short = list(zip(plates, speeds[:2]))     # 第二个序列只有 2 个
print("最短截断后：", short, "共", len(short), "条")

**预期输出**：

```text
配对记录： [('京A12345', 62, 2), ('京B67890', 132, 3), ('津C24680', 78, 1), ('冀D11223', 150, 2)]
车牌→车速字典： {'京A12345': 62, '京B67890': 132, '津C24680': 78, '冀D11223': 150}
长度校验： True
最短截断后： [('京A12345', 62), ('京B67890', 132)] 共 2 条
```

### 7.5.5 实操步骤（上机跟随）

1. `list(zip(plates, speeds, lanes))` 得到 4 个三元组，每个按下标把车牌、车速、车道对齐
2. `dict(zip(plates, speeds))` 得到 4 个键值对的字典，验证 zip 产物可直接喂给 `dict()`
3. `len(plates) == len(speeds) == len(lanes)` 输出 `True`，这是防止静默截断的前置校验
4. 把 `speeds[:2]` 传入 zip，只得到 2 条记录且无任何报错，验证以最短序列为准
5. 直接 `print(zip(...))` 会显示 `<zip object …>`，必须 `list()` 才能看到内容

### 7.5.6 常见问题 FAQ

| 现象 | 根因 | 处理 |
|---|---|---|
| 记录少了几条却没有任何报错 | 两序列长度不一致，zip 以最短为准静默截断 | 配对前先 `len(a) == len(b)` 校验 |
| 打印出 `<zip object at 0x…>` | zip 返回迭代器，未物化 | 用 `list(zip(...))` 或 `dict(zip(...))` 转换 |
| 复用同一个 zip 对象第二次为空 | 迭代器一次性消费 | 转成列表保存后再多次使用 |
| 以为是两两组合，结果只有 4 条 | zip 是对齐配对，不是笛卡尔积 | 要组合用嵌套 `for` |

## 7.6 知识点 7.5　装饰器与高阶函数

### 7.6.1 核心概念

**高阶函数**：参数或返回值是函数的函数。`map` / `filter` / `sorted` 都是高阶函数；装饰器是"接收函数、返回新函数"的高阶函数。

**装饰器的作用**：在不修改原函数代码的前提下，为它增加统一的前后处理（日志、计时、权限、缓存）。

```text
def log_call(func):            # 接收原函数
    def wrapper(*args, **kwargs):   # 新函数，包住原函数
        ...前置处理...
        return func(*args, **kwargs)   # 调用原函数
    return wrapper             # 返回新函数

@log_call
def calc_density(...): ...
```

`@log_call` 只是语法糖，等价于 `calc_density = log_call(calc_density)`。

**原理补充（概念深化）**：装饰器能「不改原函数加行为」，根源是函数为一等对象：`def` 把函数对象绑定到名字，而 `@log_call` 在定义完成后立刻执行 `calc_density = log_call(calc_density)`，把名字重新绑定到装饰器返回的新函数 `wrapper`。闭包让 `wrapper` 记住外层的 `func`，即便 `log_call` 已返回，原函数仍被引用着；此后调用 `calc_density(...)` 实际是在调 `wrapper`，先做前置处理再 `func(*args, **kwargs)` 转交。`wrapper` 必须用可变参数原样「打包再拆包透传」，才能适配任意被装饰函数的签名；且必须 `return func(...)`，否则原函数结果在 wrapper 里被丢弃、调用方拿到 `None`。常见误解是以为 `@` 有什么魔法，它只是赋值语法糖；被装饰后 `func.__name__` 变成 `wrapper`，也正说明「函数名」只是引用，不是函数对象的固有身份。

### 7.6.2 语法要点

| 要素 | 说明 |
|---|---|
| `func` | 被装饰的原函数，作为参数传入 |
| `wrapper(*args, **kwargs)` | 必须用可变参数"原样透传"，才能适配任意签名 |
| `return func(*args, **kwargs)` | 必须返回原函数结果，否则调用方拿到 `None` |
| `wrapper.calls = 0` | 给函数对象挂属性，用于计数（也可用闭包变量 + `nonlocal`） |
| `func.__name__` | 原函数名；不加 `functools.wraps` 时被装饰函数的 `__name__` 会变成 `wrapper` |

### 7.6.3 常见用法与易错点

| 易错点 | 现象 | 修正 |
|---|---|---|
| `wrapper` 写死参数 | 只能装饰固定签名的函数 | 用 `*args, **kwargs` |
| 忘记 `return func(...)` | 原函数返回值丢失，得到 `None` | 加 `return` |
| 忘记 `return wrapper` | 被装饰函数变成 `None`，调用报 `TypeError` | 装饰器必须返回函数 |
| 计数器用全局变量 | 多个装饰器互相污染 | 挂在 `wrapper` 上 |
| 装饰器带参数时漏一层 | `TypeError` | 带参装饰器需三层嵌套（本章不要求） |

---

### 7.6.4 随堂示例：装饰器：不改原函数就加日志

**核心要点**：验证装饰器的工作方式：`@log_call` 把函数替换为 `wrapper`，调用时先执行附加逻辑再执行原函数；计数器挂在 `wrapper` 属性上，因此能跨多次调用累计。

In [ ]:
def log_call(func):
    """记录被装饰函数被调用的次数"""
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        print("[日志] 第", wrapper.calls, "次调用", func.__name__, "参数", args)
        return func(*args, **kwargs)
    wrapper.calls = 0
    return wrapper

@log_call
def density(cars, length_km, lanes):
    return round(cars / (length_km * lanes), 2)

print("结果：", density(96, 2.5, 3))
print("结果：", density(120, 3, 2))
print("累计调用次数：", density.calls)

**预期输出**：

```text
[日志] 第 1 次调用 density 参数 (96, 2.5, 3)
结果： 12.8
[日志] 第 2 次调用 density 参数 (120, 3, 2)
结果： 20.0
累计调用次数： 2
```

**变式代码（迁移练习）**：相对随堂示例的计数型装饰器，本变式迁移为「打印透传参数与被装饰后函数名」的观察型装饰器，验证 `*args/**kwargs` 原样透传、必须 `return func(...)`、以及 `@` 把名字重绑到 wrapper 后 `__name__` 变为 `wrapper`。

In [ ]:
def deco(func):                                  # 高阶函数：接收原函数
    def wrapper(*a, **kw):                       # 新函数，包住原函数
        print("前置：", func.__name__, a, kw)
        return func(*a, **kw)                    # 必须返回原函数结果
    return wrapper
@deco                                            # 语法糖：等价于 fee = deco(fee)
def fee(km, rate=0.5, discount=1.0):
    return round(km * rate * discount, 2)
print("费用：", fee(120, discount=0.9))
print("被装饰后名字：", fee.__name__)

**预期输出**：

```text
前置： fee (120,) {'discount': 0.9}
费用： 54.0
被装饰后名字： wrapper
```

### 7.6.5 实操步骤（上机跟随）

1. 调用 `fee(120, discount=0.9)` 先打印 `前置： fee (120,) {'discount': 0.9}`，说明位置实参进 `a`、关键字实参进 `kw`
2. wrapper 内 `return func(*a, **kw)` 拆包转交原函数，输出 `费用： 54.0`，验证返回值没被吞掉
3. 打印 `fee.__name__` 得到 `wrapper`，验证 `@deco` 等价于 `fee = deco(fee)`、名字已指向新函数
4. 把 `return func(*a, **kw)` 改成只写 `func(*a, **kw)`，调用会得到 `None`，验证必须返回原函数结果
5. 把 `wrapper(*a, **kw)` 写死成 `wrapper(km)`，再用 `discount=` 调用会报 `TypeError`，验证必须用可变参数透传

### 7.6.6 常见问题 FAQ

| 现象 | 根因 | 处理 |
|---|---|---|
| 调用被装饰函数得到 `None` | wrapper 里漏了 `return func(...)`，结果被丢弃 | 在 wrapper 内 `return func(*args, **kwargs)` |
| `TypeError: got an unexpected keyword argument` | wrapper 参数写死，无法接收关键字实参 | 用 `wrapper(*args, **kwargs)` 原样透传 |
| 被装饰后 `__name__` 变成 `wrapper` | `@` 已把名字重绑到新函数对象 | 知道名字只是引用即可；保留元信息可用 `functools.wraps`（进阶） |
| 多个被装饰函数计数互相串味 | 计数器用了全局变量，被所有 wrapper 共享 | 把计数挂在各自的 `wrapper.calls` 上 |
| 被装饰函数变成 `None`、调用报 `TypeError` | 装饰器末尾忘记 `return wrapper` | 装饰器必须返回新函数对象 |

# 第二部分　代码示范

## 案例 7-1　车载车速数据批处理

- **覆盖知识点**：7.2（`map` / `filter`）、7.1（`lambda`）、7.3（`sorted(key=)`）
- **源文件**：`case1_车速批处理.py`
- **难度**：基础　**建议用时**：25 分钟

### 任务情境

一批车辆的车速数据需要：① 统一换算为 m/s；② 筛出超速（>120 km/h）车辆；③ 按车速从高到低排名；④ 计算超速率。要求用函数式写法，不写显式循环。

### 任务要求

1. 用 `map()` + `lambda` 完成单位换算，结果转成列表；
2. 用 `filter()` + `lambda` 筛选超速车辆；
3. 用 `sorted(key=..., reverse=True)` 排名；
4. 用 `len()` 计算超速率并保留 1 位小数。

### 参考代码

In [ ]:
# ---------- 第 07 章 高级函数 · 案例 7-1：车载车速数据批处理 ----------
# 对应知识点：7.1 lambda 匿名函数、7.2 map() 与 filter()、7.3 sorted(key=) 排序
# 功能说明：批量把车速换算为 m/s、筛出超速车辆、按车速降序排名并计算超速率

speeds = [62, 118, 45, 132, 78, 95, 150]

# 1) map：把 km/h 换算成 m/s（除以 3.6）
ms = list(map(lambda v: round(v / 3.6, 1), speeds))
print("换算 m/s：", ms)

# 2) filter：保留大于 120 的超速车辆
over = list(filter(lambda v: v > 120, speeds))
print("超速车速：", over)
print("超速车辆数：", len(over))

# 3) sorted + key：按车速降序排名（原列表不变）
ranked = sorted(speeds, key=lambda v: v, reverse=True)
print("降序排名：", ranked)
print("原列表未被修改：", speeds)

# 4) 组合计算超速率
rate = len(over) / len(speeds) * 100
print("超速率：", round(rate, 1), "%")

# 5) 等价写法对照：map 也可直接传内置函数
print("全部转字符串：", list(map(str, speeds)))

### 运行结果

```text
换算 m/s： [17.2, 32.8, 12.5, 36.7, 21.7, 26.4, 41.7]
超速车速： [132, 150]
超速车辆数： 2
降序排名： [150, 132, 118, 95, 78, 62, 45]
原列表未被修改： [62, 118, 45, 132, 78, 95, 150]
超速率： 28.6 %
全部转字符串： ['62', '118', '45', '132', '78', '95', '150']
```

### 代码解读

| 代码段 | 对应知识点 | 作用 |
|---|---|---|
| `lambda v: round(v / 3.6, 1)` | 7.1、7.2 | 就地定义"换算"函数，无需提前 `def` |
| `list(map(...))` | 7.2 | `map` 返回迭代器，转列表才可打印、可复用 |
| `filter(lambda v: v > 120, ...)` | 7.2 | 条件为真的元素保留，个数会减少 |
| `sorted(..., key=..., reverse=True)` | 7.3 | 返回新列表，原列表不变（下一行即验证） |
| `len(over) / len(speeds) * 100` | 第 2 章 | 超速率 = 超速数 ÷ 总数 |
| `map(str, speeds)` | 7.2 | `f` 可以是内置函数，不必写 `lambda` |

**改一改**：把 `sorted` 那行的 `key=lambda v: v` 删掉，看结果是否相同（元素本身即键时 `key` 可省）；再把 `filter` 换成 `map`，观察"筛选"与"变换"的区别。

## 案例 7-2　用装饰器给采集函数加统一日志

- **覆盖知识点**：7.5（装饰器、`*args` / `**kwargs` 透传、`@` 语法糖）、第 6 章（嵌套函数）
- **源文件**：`case2_装饰器日志.py`
- **难度**：进阶　**建议用时**：35 分钟

### 任务情境

数据采集函数有十几个，运维要求每次调用都记录"被调函数名、参数、累计调用次数"，但不允许逐个修改函数体。

### 任务要求

1. 编写装饰器 `log_call(func)`，内部定义 `wrapper(*args, **kwargs)`；
2. `wrapper` 内先累加调用次数，再打印日志，最后返回原函数结果；
3. 用 `@log_call` 装饰两个不同签名的函数（一个位置参数、一个关键字参数）；
4. 说明为什么 `wrapper` 必须用 `*args, **kwargs` 透传。

### 参考代码

In [ ]:
# ---------- 第 07 章 高级函数 · 案例 7-2：用装饰器给采集函数加统一日志 ----------
# 对应知识点：7.5 装饰器与 @ 语法糖、嵌套函数与闭包、*args / **kwargs 透传
# 功能说明：为多个采集函数统一加上调用日志与调用次数统计，不修改原函数体内部代码

def log_call(func):
    """装饰器：记录被装饰函数的调用次数与参数"""
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        print("[日志] 第 " + str(wrapper.calls) + " 次调用 " + func.__name__
              + " 参数=" + str(args) + str(kwargs))
        return func(*args, **kwargs)
    wrapper.calls = 0          # 给 wrapper 挂一个计数器属性
    return wrapper


@log_call
def calc_density(cars, length_km, lanes):
    return round(cars / (length_km * lanes), 2)


@log_call
def calc_fee(km, rate=0.5, discount=1.0):
    return round(km * rate * discount, 2)


print("密度结果：", calc_density(96, 2.5, 3))
print("密度结果：", calc_density(120, 3.0, 2))
print("费用结果：", calc_fee(120, discount=0.95))
print("累计调用 calc_density：", calc_density.calls)
print("累计调用 calc_fee：", calc_fee.calls)

### 运行结果

```text
[日志] 第 1 次调用 calc_density 参数=(96, 2.5, 3){}
密度结果： 12.8
[日志] 第 2 次调用 calc_density 参数=(120, 3.0, 2){}
密度结果： 20.0
[日志] 第 1 次调用 calc_fee 参数=(120,){'discount': 0.95}
费用结果： 57.0
累计调用 calc_density： 2
累计调用 calc_fee： 1
```

### 代码解读

| 代码段 | 对应知识点 | 作用 |
|---|---|---|
| `def log_call(func):` | 7.5 | 接收函数、返回函数，属高阶函数 |
| `def wrapper(*args, **kwargs):` | 7.5、6.2 | 原样接收任意参数：位置参数进 `args` 元组，关键字参数进 `kwargs` 字典 |
| `wrapper.calls += 1` | 7.5 | 计数挂在函数对象上，两个被装饰函数各有一份 |
| `func.__name__` | 7.5 | 取原函数名用于日志 |
| `return func(*args, **kwargs)` | 7.5 | 拆包透传并**返回**结果，缺 `return` 会导致结果为 `None` |
| `wrapper.calls = 0` | 7.5 | 在 `return wrapper` 之前初始化属性 |
| `@log_call` | 7.5 | 语法糖，等价 `calc_density = log_call(calc_density)` |
| `calc_density.calls` | 7.5 | 装饰后 `calc_density` 实际指向 `wrapper`，故能读到计数 |

**改一改**：把 `wrapper` 的参数改成 `(a, b, c)`，再调用 `calc_fee(120, discount=0.95)`，记录报错信息——这就是"必须用 `*args, **kwargs`"的直接原因。

---

# 第三部分　布置练习

## 练习 7-1　车牌与车速配对筛选

- **对应知识点**：7.4（`zip`）、7.2（`filter` / `map`）、7.1（`lambda`）
- **提交文件**：`practice1_超速配对.py`
- **难度**：基础　**建议用时**：25 分钟

### 题目

现有车牌列表与对应车速列表，需要配对后筛选出超速车辆，并输出超速车牌清单。

### 要求

1. 用 `zip()` 把两个列表配对成 `(车牌, 车速)` 的列表；
2. 用 `filter()` + `lambda` 筛出车速大于 120 的记录；
3. 用 `map()` + `lambda` 只提取超速车牌；
4. 输出超速数量与占比（保留 1 位小数）。

### 提示

- `zip` 的配对数以**较短列表**为准，两个列表长度不一致时会静默截断，实际项目中应先 `len()` 校验。
- 配对后每个元素是元组，取车速用 `p[1]`、取车牌用 `p[0]`，不要写错下标。
- 三步操作都是"返回迭代器 → `list()` 转换"，漏转会打印出对象地址。

### 自检要点（不含答案）

- [ ] 配对结果 5 条，每条是含两个元素的元组；
- [ ] 超速记录 2 条，超速车牌为 `['京C24680', '京E55667']`（顺序按原列表）；
- [ ] 超速占比 40.0%；
- [ ] 有 `len(plates) == len(speeds)` 的校验输出。

## 练习 7-2　路段多关键字排序

- **对应知识点**：7.3（`sorted(key=)`、元组 key）、7.1（`lambda`）、7.2（`map`）
- **提交文件**：`practice2_路段排序.py`
- **难度**：综合　**建议用时**：25 分钟

### 题目

路网中各路段数据为 `(路段名, 平均车速, 排队长度)`，需要按不同业务口径排序：① 排队长度降序（拥堵优先处置）；② 先按车速升序、车速相同再按排队降序。

### 要求

1. 用 `sorted(key=lambda r: r[2], reverse=True)` 实现排队降序；
2. 用元组作为 key 实现二级排序 `key=lambda r: (r[1], -r[2])`；
3. 用 `map` 提取排序后的路段名；
4. 输出两种排序结果并对比。

### 提示

- `sorted` 的 `key` 返回元组时，Python 按元组元素依次比较；用 `-r[2]` 可在整体升序中实现"第二列降序"。
- 负号只能用于数值字段；若按字符串字段做二级降序，需分两次 `sorted`（稳定排序，第二次为主键）。
- 提取路段名用 `map(lambda r: r[0], ...)`，结果需 `list()`。

### 自检要点（不含答案）

- [ ] 排队降序结果为京港澳(320)、京承(120)、京开(80)、机场(40)；
- [ ] 二级排序中"京港澳"与"京承"车速同为 45，排在前者因排队更长；
- [ ] 处置顺序列表 4 个元素，顺序与排序结果一致；
- [ ] 两种排序都不修改原列表（可对比输出）。

---

# 第四部分　习题答案

## 答案 7-1　车牌与车速配对筛选

> 对应 **练习 7-1**（第三部分）

### 参考代码

In [ ]:
# ---------- 第 07 章 高级函数 · 练习 7-1 参考答案：车牌与车速配对筛选 ----------
# 对应知识点：7.4 zip() 配对、7.2 filter() 与 map()、7.1 lambda
# 功能说明：把车牌列表与车速列表配对，筛出超速记录并提取对应车牌，计算超速占比

plates = ["京A12345", "京B67890", "津C24680", "冀D11223", "京E55667"]
speeds = [62, 118, 132, 78, 150]

pairs = list(zip(plates, speeds))
print("配对结果：", pairs)

over = list(filter(lambda p: p[1] > 120, pairs))
print("超速记录：", over)
print("超速数量：", len(over))

over_plates = list(map(lambda p: p[0], over))
print("超速车牌：", over_plates)

print("超速占比：", round(len(over) / len(pairs) * 100, 1), "%")
print("车牌数校验一致：", len(plates) == len(speeds))

### 运行结果

```text
配对结果： [('京A12345', 62), ('京B67890', 118), ('津C24680', 132), ('冀D11223', 78), ('京E55667', 150)]
超速记录： [('津C24680', 132), ('京E55667', 150)]
超速数量： 2
超速车牌： ['津C24680', '京E55667']
超速占比： 40.0 %
车牌数校验一致： True
```

### 解析

1. **配对的意义**：车牌与车速分属两个列表时，"第 3 辆车超速"这类结论无法直接表达；`zip` 之后每条记录自带车牌，筛选结果可直接读。
2. **两步取字段**：`filter` 的判断函数用 `p[1]`（车速），`map` 的提取函数用 `p[0]`（车牌）。下标与业务字段的对应关系必须写清注释，否则极易写反。
3. **占比计算**：`len(over) / len(pairs)`，分母用配对后的长度（等于较短列表长度），与 `len(plates)` 在长度一致时相同。
4. **长度校验的必要性**：若两个列表不等长，`zip` 会静默截断，导致"少统计了几辆车"而无任何报错。校验输出 `True` 是给读者的信心，也是给自己留的排错线索。

### 常见错误

| 错误现象 | 原因 | 修正 |
|---|---|---|
| `<zip object at 0x…>` | 未 `list()` 转换 | `list(zip(...))` |
| 超速车牌取到车速 | 混淆 `p[0]` / `p[1]` | 车牌 `p[0]`、车速 `p[1]` |
| 超速占比 66.7% | 分母用了 `len(over)` | 分母为总记录数 |
| 结果只有 3 条 | 两列表长度不一致 | 先校验长度 |

### 评分要点（供批改）

| 项 | 分值 | 观察点 |
|---|---|---|
| `zip` 配对 | 3 | 结果为 5 个二元组 |
| `filter` 筛选 | 3 | 阈值 120、结果 2 条 |
| `map` 提取 | 2 | 得到车牌而非元组 |
| 占比与校验 | 2 | 40.0%、长度校验为 `True` |

## 答案 7-2　路段多关键字排序

> 对应 **练习 7-2**（第三部分）

### 参考代码

In [ ]:
# ---------- 第 07 章 高级函数 · 练习 7-2 参考答案：路段多关键字排序 ----------
# 对应知识点：7.3 sorted(key=) 的多级 key（元组 + 负号取反）、7.1 lambda、7.2 map()
# 功能说明：按排队长度降序排出处置顺序，并按「车速升序、同速排队降序」做多关键字排序

roads = [
    ("京港澳", 45, 320),
    ("京开", 62, 80),
    ("机场", 78, 40),
    ("京承", 45, 120),
]

by_queue = sorted(roads, key=lambda r: r[2], reverse=True)
print("按排队降序：", by_queue)
print("处置顺序：", list(map(lambda r: r[0], by_queue)))

by_speed = sorted(roads, key=lambda r: (r[1], -r[2]))
print("按车速升序、同速排队降序：", by_speed)

print("最拥堵路段：", by_queue[0][0], "排队", by_queue[0][2], "m")
print("平均车速：", round(sum(map(lambda r: r[1], roads)) / len(roads), 1))

### 运行结果

```text
按排队降序： [('京港澳', 45, 320), ('京承', 45, 120), ('京开', 62, 80), ('机场', 78, 40)]
处置顺序： ['京港澳', '京承', '京开', '机场']
按车速升序、同速排队降序： [('京港澳', 45, 320), ('京承', 45, 120), ('京开', 62, 80), ('机场', 78, 40)]
最拥堵路段： 京港澳 排队 320 m
平均车速： 57.5
```

### 解析

1. **单键排序**：`key=lambda r: r[2]` 取排队长度，`reverse=True` 让最长者在前，得到"处置优先级"。
2. **元组 key 的比较规则**：`(r[1], -r[2])` 先生比车速（升序），车速相同时比较 `-排队`（负号把"降序"转成"升序"下的比较），因此京港澳（-320）排在京承（-120）之前。
3. **两种排序结果相同是巧合**：本例中车速与排队恰好呈反相关，故两种顺序一致。若加入一条 `("京新", 45, 500)`，二级排序会把它排在最前，而排队降序也会——真正的差异要构造"车速低但排队短"的数据才能看出。
4. **`map` 提取路段名**：`list(map(lambda r: r[0], by_queue))` 得到可直接展示的处置清单，比打印元组更贴近业务。
5. **平均车速用 `map`**：`sum(map(lambda r: r[1], roads))` 与推导式 `sum(r[1] for r in roads)` 等价，本章按函数式写法统一。

### 常见错误

| 错误现象 | 原因 | 修正 |
|---|---|---|
| `TypeError: bad operand type for unary -` | 对字符串用负号 | 负号只用于数值字段 |
| 二级排序第二列仍升序 | `key=lambda r: (r[1], r[2])` | 加负号 `-r[2]` |
| 原列表被改 | 用了 `roads.sort(...)` | 用 `sorted()` 返回新列表 |
| `map` 结果无法打印 | 未 `list()` | 转换后再输出 |

### 评分要点（供批改）

| 项 | 分值 | 观察点 |
|---|---|---|
| 单键降序 | 3 | 排队从 320 到 40 |
| 元组 key 二级排序 | 4 | 同速时按排队降序 |
| `map` 提取 | 2 | 处置顺序列表正确 |
| 未改原列表 | 1 | `sorted` 而非 `sort` |

---

## 本章小结

- **`lambda`**：单表达式匿名函数，无 `return`，只用于"一次性小逻辑"。
- **`map` / `filter`**：变换 / 筛选，返回**迭代器**，需 `list()` 转换且只能遍历一次。
- **`sorted(key=)`**：`key` 传函数本身（不加括号）；元组 key 实现多级排序，负号实现"某列降序"。
- **`zip`**：按下标配对，以最短序列为准，静默截断，配对前先校验长度。
- **装饰器**：接收函数、返回新函数；`wrapper(*args, **kwargs)` 透传并 `return` 原函数结果；`@` 是语法糖。

## 自检清单

- [ ] 能写出 `lambda x: x * 2` 并说明它不能含语句；
- [ ] 知道 `map` / `filter` 返回迭代器，能解释为何要 `list()`；
- [ ] 会写单键与元组 key 两种 `sorted`；
- [ ] 知道 `zip` 长度不一致会怎样，如何预防；
- [ ] 能默写一个"计数型"装饰器，并解释 `*args` `**kwargs` 的必要性；
- [ ] 能说明 `@log_call` 与 `f = log_call(f)` 的等价关系。

## 前后章节衔接

- 本章的 `sorted(key=...)` 将在**第 8 章练习**用于"按小时汇总后取峰值"、在**第 9 章考核**用于"流量最大时段"。
- 本章的装饰器思路（前后包裹）将在**第 11 章**与属性封装（`set_green` 校验）形成对照：一个包裹函数、一个包裹赋值。
- 本章用 `len()` / `sum()` 处理 `map` 结果，**第 10 章**的 `random` 与 `datetime` 会与它们配合生成模拟数据集。

---

[🏠 课程首页](../00_课程索引.ipynb)&nbsp;&nbsp;|&nbsp;&nbsp;[◀ 上一章：函数](../06函数/06_函数.ipynb)&nbsp;&nbsp;|&nbsp;&nbsp;[下一章：文件操作 ▶](../08文件操作/08_文件操作.ipynb)